# 01 — ChainPilot Data Exploration

Exploratory analysis of the three client Excel datasets.

**Sources:**
- `Raw_data/Stock management.xlsx` — board material stock ledger
- `Raw_data/GLUE RECORD-1.xlsx` — glue consumable ledger
- `Raw_data/Tree log suppliers Book.xlsx` — supplier delivery records

Run the ETL scripts first to generate catalog and quality reports:
```bash
python -m etl.ingest_catalog
python -m etl.profile_data
```

In [ ]:
import os
import json
import re
from collections import defaultdict
from decimal import Decimal, InvalidOperation

import openpyxl
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DATA = os.path.join(ROOT, 'Raw_data')
DATA_DIR = os.path.join(ROOT, 'data')

print('Root:', ROOT)
print('Raw data files:', os.listdir(RAW_DATA))

## 1. Load Catalog

In [ ]:
catalog_path = os.path.join(DATA_DIR, 'catalog', 'catalog.json')
if os.path.exists(catalog_path):
    with open(catalog_path) as f:
        catalog = json.load(f)
    for src in catalog['sources']:
        print(f"\n{src['source_id']} — {src['filename']} ({src.get('file_size_kb', '?')} KB)")
        for sheet in src.get('sheets', []):
            print(f"  [{sheet['sheet_name']}] rows={sheet['total_rows']} cols={sheet['column_count']}")
else:
    print('Catalog not found — run: python -m etl.ingest_catalog')

## 2. Stock Management — Tab Overview

In [ ]:
THICKNESS_RE = re.compile(r'^\d+\s*[Mm]{2}\s*$')

wb_stock = openpyxl.load_workbook(os.path.join(RAW_DATA, 'Stock management.xlsx'), data_only=True)
print(f'All sheets ({len(wb_stock.sheetnames)}): {wb_stock.sheetnames}')

valid_tabs = [s for s in wb_stock.sheetnames if THICKNESS_RE.match(s)]
print(f'\nValid thickness tabs ({len(valid_tabs)}): {valid_tabs}')
print(f'Skipped: {[s for s in wb_stock.sheetnames if not THICKNESS_RE.match(s)]}')

## 3. Stock Management — Items and Specifications by Tab

In [ ]:
rows = []
for tab in valid_tabs:
    ws = wb_stock[tab]
    tab_rows = list(ws.iter_rows(values_only=True))
    # Find header row
    header_idx = next(
        (i for i, r in enumerate(tab_rows)
         if any(str(c).strip().lower() == 'items' for c in r if c)),
        -1
    )
    if header_idx < 0:
        rows.append({'tab': tab, 'items': None, 'specs': [], 'error': 'no header'})
        continue
    
    data = tab_rows[header_idx + 1:]
    first_items = next((str(r[1]).strip() for r in data if len(r) > 1 and r[1]), None)
    all_specs = list({str(r[2]).strip() for r in data if len(r) > 2 and r[2] and not str(r[2]).startswith('=')})
    
    rows.append({
        'tab': tab.strip(),
        'items_value': first_items,
        'display_name': f'{first_items} MDF' if first_items else None,
        'spec_variants': all_specs,
        'variant_count': len(all_specs),
        'canonical_spec': all_specs[0] if all_specs else None,
    })

df = pd.DataFrame(rows)
df

## 4. Parse Specification Strings

In [ ]:
def parse_spec(raw):
    parts = raw.split('*')
    if len(parts) != 3:
        return None, f'expected 3 segments, got {len(parts)}'
    try:
        return {'thickness': Decimal(parts[0]), 'width': Decimal(parts[1]), 'length': Decimal(parts[2])}, None
    except InvalidOperation as e:
        return None, str(e)

parsed_rows = []
for _, row in df.iterrows():
    if not row['canonical_spec']:
        continue
    dims, err = parse_spec(row['canonical_spec'])
    parsed_rows.append({
        'tab': row['tab'],
        'display_name': row['display_name'],
        'raw_spec': row['canonical_spec'],
        **({} if err else {'thickness': float(dims['thickness']),
                            'width': float(dims['width']),
                            'length': float(dims['length'])}),
        'parse_error': err,
    })

pd.DataFrame(parsed_rows)

## 5. Supplier Workbook — Received Tree Logs Headers

In [ ]:
wb_sup = openpyxl.load_workbook(os.path.join(RAW_DATA, 'Tree log suppliers Book.xlsx'), data_only=True)
print('All sheets:', wb_sup.sheetnames[:5], '...')

ws_trucks = wb_sup['Received tree logs trucks']
for i, row in enumerate(ws_trucks.iter_rows(values_only=True)):
    cells = [str(c).strip() for c in row if c is not None and str(c).strip()]
    if len(cells) >= 2:
        print(f'\nHeader row (row {i+1}):')
        for j, h in enumerate(cells):
            print(f'  [{j}] {h!r}')
        break

## 6. GLUE RECORD-1 — Overview

In [ ]:
wb_glue = openpyxl.load_workbook(os.path.join(RAW_DATA, 'GLUE RECORD-1.xlsx'), data_only=True)
print('Sheets:', wb_glue.sheetnames)

ws_glue = wb_glue['GLUE A , B, C and D']
glue_rows = list(ws_glue.iter_rows(values_only=True))[:5]
for i, row in enumerate(glue_rows):
    print(f'Row {i+1}: {row}')

## 7. Data Quality Report

In [ ]:
report_path = os.path.join(DATA_DIR, 'profiles', 'data_quality_report.json')
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    for src in report['sources']:
        print(f"\n{'='*40}")
        print(f"{src['source_id']} — total_issues={src.get('total_issues', '?')}")
        for sheet in src.get('sheets', []):
            issues = sheet.get('issues', [])
            if issues:
                print(f"  [{sheet['sheet_name']}]")
                for issue in issues:
                    print(f"    >> {issue}")
else:
    print('Quality report not found — run: python -m etl.profile_data')